<h1 align = "Center">
Lotus Group Customer Value & Profitability Analysis

<h2 align = "Center">
Data Cleaning and Integration
</h1>

In [1]:

import pandas as pd
from pathlib import Path

# Define path to folder
data_path = Path("../01_Data/raw")

def load_csv (filename):
    return pd.read_csv(data_path/filename)

# Customer Records Cleaning

We want to establish a reliable customer dimension. We already identified:

- 3,050 customer records.

- 50 duplicated records, representing 50 customers appearing twice.

- 3,000 distinct customer IDs.

- 567 invalid birth dates.

- 382 missing email addresses.

- Inconsistent capitalization in the gender column.

We start by creatinga  working copy

In [2]:
# Preserve the original dataset and create a working copy
customers = load_csv("tbl_customers.csv")
customers_clean = customers.copy()

print("Original records:", len(customers))
print("Working copy records:", len(customers_clean))

Original records: 3050
Working copy records: 3050


We've established earlier that the 50 duplicated customer records are exact duplicates, not conflicting customer profiles.

Therefore, retaining one copy of each record is appropriate.

What we expect:
- Records after removing duplicates: 3,000.
- Remaining duplicate customer IDs: 0.

Why are we doing this?

We expect our analysis to eventually calculate customer-level metrics such as:
- Total historical spending.
- Number of purchases.
- Average order value.
- Purchase frequency.
- Customer tenure.

If a customer appears more than once in the customer dimension, joining that dimension to transaction records could duplicate transactions and inflate these metrics.

Removing exact duplicates prevents that problem.

In [3]:
# Remove exact duplicate records
customers_clean = customers_clean.drop_duplicates()

print("Records after removing duplicates:", len(customers_clean))

print(
    "Remaining duplicate customer IDs:",
    customers_clean["customer_id"].duplicated().sum()
)

Records after removing duplicates: 3000
Remaining duplicate customer IDs: 0


## Standardizing Customer Gender

We've identified six variations:

`Male`, `male`, `MALE`, `Female`, `female`, `FEMALE`

These represent only two actual categories. We'll standardize capitalization to avoid treating them as separate categories during analysis.

**Why it matters**

When we profile customer segments, we may want to compare spending and purchasing frequency across demographic groups.

Standardization ensures that all customers belonging to the same category are grouped correctly.

In [4]:
# Standardize gender capitalization and remove surrounding whitespace
customers_clean["gender"] = (
    customers_clean["gender"]
    .str.strip()
    .str.capitalize()
)

# Verify the results
print(customers_clean["gender"].value_counts(dropna=False))

gender
Male      1669
Female    1331
Name: count, dtype: int64


## Handling invalid Birth Dates

We previously discovered 567 invalid birth dates.

We want to convert the column into a proper datetime field. Invalid entries will automatically become missing values.

In [5]:
# Convert birth_date to datetime
customers_clean["birth_date"] = pd.to_datetime(
    customers_clean["birth_date"],
    errors="coerce"
)

# Check the results
print("Birth date data type:", customers_clean["birth_date"].dtype)

print(
    "Missing/invalid birth dates:",
    customers_clean["birth_date"].isna().sum()
)

Birth date data type: datetime64[us]
Missing/invalid birth dates: 560


We are deliberately not imputing these missing dates because there is no reliable basis for reconstructing a customer's date of birth.

## Convert Registration Dates and Phone Numbers

We want to convert `registration_date` to datetime so we can calculate customer tenure later.

We also want to convert `phone` to text because phone numbers are identifiers, not quantities. We should not perform mathematical operations on them.

In [6]:
# Convert registration_date to datetime
customers_clean["registration_date"] = pd.to_datetime(
    customers_clean["registration_date"],
    errors="coerce"
)

# Convert phone numbers to strings
customers_clean["phone"] = (
    customers_clean["phone"]
    .astype("string")
)

# Verify data types
print(customers_clean[["registration_date", "phone"]].dtypes)

# Check for missing values
print("\nMissing values:")
print(
    customers_clean[
        ["registration_date", "phone", "email"]
    ].isna().sum()
)

registration_date    datetime64[us]
phone                        string
dtype: object

Missing values:
registration_date      0
phone                  0
email                374
dtype: int64


So we are retaining missing emails and phone values because we don't need to discard customer records simply because their contact information is incomplete.

Our objective is to understand customer value and profitability, not to contact customers.

Missing contact information does not prevent us from calculating historical spending, purchase frequency or other behavioural features.

## Final Customer Dimension Verification

In [7]:
print("CUSTOMER DIMENSION SUMMARY")
print("--------------------------")

print("Total records:", len(customers_clean))
print("Unique customer IDs:", customers_clean["customer_id"].nunique())
print("Duplicate customer IDs:", customers_clean["customer_id"].duplicated().sum())

print("\nData types:")
print(customers_clean.dtypes)

print("\nMissing values:")
print(customers_clean.isna().sum())

CUSTOMER DIMENSION SUMMARY
--------------------------
Total records: 3000
Unique customer IDs: 3000
Duplicate customer IDs: 0

Data types:
customer_id                     str
full_name                       str
gender                          str
birth_date           datetime64[us]
phone                        string
email                           str
city                            str
region                          str
loyalty_tier                    str
registration_date    datetime64[us]
dtype: object

Missing values:
customer_id            0
full_name              0
gender                 0
birth_date           560
phone                  0
email                374
city                   0
region                 0
loyalty_tier           0
registration_date      0
dtype: int64


Let's save the cleaned dataset

In [8]:
# Save customers_clean datset to processed folder
customers_clean.to_csv("../01_Data/processed/customers_clean.csv")

# Orders Table Cleaning and Preparation

This table will provide the foundation for our customer value analysis and segmentation.

We want to start by consolidating the two source tables:

- orders_2022_2023
- orders_2024

The quality check showed us that the consolidated table should contain 12,000 unique orders.

In [9]:
# Loading order tables separately for consolidation
orders_2022_2023 = load_csv("tbl_orders_2022_2023.csv")
orders_2024 = load_csv("tbl_orders_2024.csv")

# Appending orders_2024 to orders_2022_2023
orders = pd.concat(
    [orders_2022_2023, orders_2024],
    ignore_index = True
)

orders_clean = orders.copy()
print("Total order records:", len(orders_clean))
print("Distinct order IDs:", orders_clean["order_id"].nunique())


Total order records: 12000
Distinct order IDs: 12000


## Standardizing Payment Methods

From the data inventory notebook, we identified leading spaces in some payment methods.

For example:

- "Cash"
- "  Cash"

These should represent the same payment method.

In [10]:
# Remove leading and trailing whitespace
orders_clean["payment_method"] = (
    orders_clean["payment_method"].str.strip()
)

# Examine the standardized categories
print(orders_clean["payment_method"].value_counts())

payment_method
Cash             2070
Vodafone Cash    1990
InstaPay         1990
Credit Card      1986
Debit Card       1984
Fawry            1980
Name: count, dtype: int64


## Converting order dates

Why these transformations are important

1. Standardized payment methods prevent artificial categories during analysis.

2. Proper datetime values allow us to calculate purchase recency, frequency and historical spending, and to split orders chronologically for model testing.

In [11]:
orders_clean["order_date"] = pd.to_datetime(
    orders_clean["order_date"],
    errors="coerce"
)

print("Order date data type:", orders_clean["order_date"].dtype)

print(
    "Missing/invalid order dates:",
    orders_clean["order_date"].isna().sum()
)

Order date data type: datetime64[us]
Missing/invalid order dates: 0


## Verigying Financial and Order Status Fields

We want to make sure our financial columns are numeric and our order statuses are consistent. Our revenue and profit figures depend on these fields. We therefore need reliable financial values and consistent order-status categories.

In [12]:
# Check financial column data types
print("Financial column data types:")
print(orders_clean[["total_revenue", "total_cost"]].dtypes)

# Check for missing financial values
print("\nMissing financial values:")
print(orders_clean[["total_revenue", "total_cost"]].isna().sum())

# Verify order statuses
print("\nOrder statuses:")
print(orders_clean["order_status"].value_counts())

# Verify negative financial values
print("\nNegative revenue:", (orders_clean["total_revenue"] < 0).sum())
print("Negative cost:", (orders_clean["total_cost"] < 0).sum())

Financial column data types:
total_revenue    float64
total_cost       float64
dtype: object

Missing financial values:
total_revenue    0
total_cost       0
dtype: int64

Order statuses:
order_status
Completed    10442
Returned      1063
Cancelled      495
Name: count, dtype: int64

Negative revenue: 0
Negative cost: 0


In [13]:
# Save orders_clean datasetto processed folder
orders_clean.to_csv("../01_Data/processed/orders_clean.csv")

# Cleaning Returns Table

Returns matter because they help us distinguish recorded sales from revenue that was subsequently refunded.

In [14]:
# Create a working copy
returns = load_csv("tbl_returns.csv")
returns_clean = returns.copy()

In [15]:
# Convert return_date into proper datetime column
returns_clean["return_date"] = pd.to_datetime(
    returns_clean["return_date"], errors="coerce"
)

### Necessary checks

We want to:
- Confirm unique return IDs to ensure each return record is identifiable.
- Check missing values and data types to ensure the table is usable.
- Inspect return statuses to distinguish completed refunds from pending or rejected ones.
- Check return amounts to identify invalid financial values.
- Validate dates to ensure we can correctly relate returns to the original orders.

We will use the results to establish a consistent treatment of returned orders when calculating revenue and profit.

In [16]:
# Necessar checks
print("Returns records:", len(returns_clean))
print("Unique return IDs:", returns_clean["return_id"].nunique())

print("\nData types:")
print(returns_clean.dtypes)

print("\nMissing values:")
print(returns_clean.isna().sum())

print("\nReturn statuses:")
print(returns_clean["return_status"].value_counts())

print("\nReturn amounts summary:")
print(returns_clean["return_amount"].describe())

print("\nNegative return amounts:",
      (returns_clean["return_amount"] < 0).sum())

print("\nInvalid return dates:",
      returns_clean["return_date"].isna().sum())

Returns records: 1056
Unique return IDs: 1056

Data types:
return_id                   str
order_id                    str
return_date      datetime64[us]
return_reason               str
return_amount           float64
refund_method               str
return_status               str
dtype: object

Missing values:
return_id        0
order_id         0
return_date      0
return_reason    0
return_amount    0
refund_method    0
return_status    0
dtype: int64

Return statuses:
return_status
Refunded    819
Pending     150
Rejected     87
Name: count, dtype: int64

Return amounts summary:
count     1056.000000
mean      3800.295928
std       8361.739872
min         76.500000
25%        458.500000
50%        961.500000
75%       2243.375000
max      95870.000000
Name: return_amount, dtype: float64

Negative return amounts: 0

Invalid return dates: 0


### Establishing how returns relate to orders

Before deciding how returns affect our revenue and profit calculations, we want to verify that the return records correspond to orders marked as Returned.

From the data qualitynotebook, we established that seven orders marked Returned have no corresponding return record. This check will confirm the relationship from the returns table's perspective.

In [17]:
# Attach the order status to each return record
return_status_check = returns_clean.merge(
    orders_clean[["order_id", "order_status"]],
    on="order_id",
    how="left"
)

print("Return records by original order status:")
print(return_status_check["order_status"].value_counts(dropna=False))

print("\nReturn records linked to non-Returned orders:",
      (return_status_check["order_status"] != "Returned").sum())

print("\nUnique orders with return records:",
      return_status_check["order_id"].nunique())

Return records by original order status:
order_status
Returned    1056
Name: count, dtype: int64

Return records linked to non-Returned orders: 0

Unique orders with return records: 1056


In [18]:
# Save returns_clean dataset to processed folder
returns_clean.to_csv("../01_Data/processed/returns_clean.csv")

# Preparing Order Details 

We need reliable transaction-line data before aggregating purchases into customer-level historical features.

In [19]:
# Load and create a copy
order_details = load_csv("tbl_order_details.csv")
order_details_clean = order_details.copy()

In [20]:
# Necessary Checks
print("Records:", len(order_details_clean))
print("Unique detail IDs:", order_details_clean["detail_id"].nunique())

print("\nData types:")
print(order_details_clean.dtypes)

print("\nMissing values:")
print(order_details_clean.isna().sum())

print("\nNegative values:")
numeric_columns = [
    "quantity", "unit_price", "discount_pct",
    "selling_price", "unit_cost",
    "line_total_revenue", "line_total_cost"
]

print((order_details_clean[numeric_columns] < 0).sum())

Records: 25099
Unique detail IDs: 25099

Data types:
detail_id                 str
order_id                  str
product_id                str
quantity                int64
unit_price              int64
discount_pct            int64
selling_price         float64
unit_cost               int64
line_total_revenue    float64
line_total_cost         int64
dtype: object

Missing values:
detail_id             0
order_id              0
product_id            0
quantity              0
unit_price            0
discount_pct          0
selling_price         0
unit_cost             0
line_total_revenue    0
line_total_cost       0
dtype: int64

Negative values:
quantity              0
unit_price            0
discount_pct          0
selling_price         0
unit_cost             0
line_total_revenue    0
line_total_cost       0
dtype: int64


#### Integrity check
We want to confirm that every order detail references an order and every product ID references a product.

In [21]:
# Check order references
unmatched_orders = ~order_details_clean["order_id"].isin(
    orders_clean["order_id"]
)

# reload products dataset
products = load_csv("tbl_products.csv")
# Check product references
unmatched_products = ~order_details_clean["product_id"].isin(
    products["product_id"]
)

print("Order details with unmatched order IDs:",
      unmatched_orders.sum())

print("Order details with unmatched product IDs:",
      unmatched_products.sum())

Order details with unmatched order IDs: 0
Order details with unmatched product IDs: 0


# Integration

Before integrating the transaction data, we want to copy the rest of the datasets that didn't need cleaning to the processed folder for when we'll want to use them. We don't want to manipulate the raw datasets so we can allow for reproducibility.

In [22]:
# Creating copies of remaining datasets
employees = load_csv("tbl_employees.csv")
employees_clean = employees.copy()

order_details = load_csv("tbl_order_details.csv")
order_details_clean = order_details.copy()

products = load_csv("tbl_products.csv")
products_clean = products.copy()

stores = load_csv("tbl_stores.csv")
stores_clean = stores.copy()

date = load_csv("tbl_date.csv")
date_clean = date.copy()

# Saving datasets
employees_clean.to_csv("../01_Data/processed/employees_clean.csv")
order_details_clean.to_csv("../01_Data/processed/order_details_clean.csv")
products_clean.to_csv("../01_Data/processed/products_clean.csv")
stores_clean.to_csv("../01_Data/processed/stores_clean.csv")
date_clean.to_csv("../01_Data/processed/date_clean.csv")

## Integrating Transaction Data

We want now to create a consolidated analytical dataset by joining:

1. orders_clean
2. order_details_clean
3. customers_clean
4. products

This should give us the information we need to investigate customer purchasing patterns, spending behaviour, product preferences and profitability, and eventually construct modelling features.

One important distinction: We want to preserve the transaction-line grain during integration. One row will represent one product line within an order — not one customer or one order. We'll aggregate to customer level later when engineering features.

In [23]:
transaction_data = order_details_clean.merge(
    orders_clean,
    on="order_id",
    how="left",
    validate="many_to_one"
)

print("Transaction records:", len(transaction_data))
print("Unique orders:", transaction_data["order_id"].nunique())
print("Unique customers:", transaction_data["customer_id"].nunique())
print("Missing customer IDs:", transaction_data["customer_id"].isna().sum())

Transaction records: 25099
Unique orders: 12000
Unique customers: 2945
Missing customer IDs: 0


The `validate="many_to_one"` argument ensures that multiple product lines can belong to one order, but each order ID must identify only one order record.

### Adding Customer Information



In [24]:
transaction_data = transaction_data.merge(
    customers_clean,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

print("Transaction records:", len(transaction_data))
print("Missing customer names:", transaction_data["full_name"].isna().sum())
print("Missing customer regions:", transaction_data["region"].isna().sum())

Transaction records: 25099
Missing customer names: 0
Missing customer regions: 0


### Adding Product Information

In [25]:
transaction_data = transaction_data.merge(
    products_clean,
    on="product_id",
    how="left",
    validate="many_to_one"
)

print("Transaction records:", len(transaction_data))

print("Missing product names:",
      transaction_data["product_name_raw"].isna().sum())

print("Missing product categories:",
      transaction_data["category"].isna().sum())

print("Missing product brands:",
      transaction_data["brand"].isna().sum())

Transaction records: 25099
Missing product names: 0
Missing product categories: 0
Missing product brands: 0


### Adding Date Dimension

In [26]:
transaction_data = transaction_data.merge(
    date_clean,
    on="date_id",
    how="left",
    validate="many_to_one"
)

print("Transaction records:", len(transaction_data))

print("Missing full dates:",
      transaction_data["full_date"].isna().sum())

print("\nOrder date range:")
print(transaction_data["order_date"].min(),
      "to",
      transaction_data["order_date"].max())

print("\nDate dimension range:")
print(transaction_data["full_date"].min(),
      "to",
      transaction_data["full_date"].max())

Transaction records: 25099
Missing full dates: 0

Order date range:
2022-01-01 00:00:00 to 2024-12-31 00:00:00

Date dimension range:
2022-01-01 to 2024-12-31


## Final Structural Check

We want to verify that:

1. Dataset dimensions: Confirms the size of our integrated dataset.
2. Transaction detail IDs: Ensures integration hasn't duplicated transaction lines.
3. Missing analytical fields: Confirms the essential identifiers, dates and financial/product fields are available.

In [27]:
print("Dataset dimensions:", transaction_data.shape)

print("\nDuplicate transaction detail IDs:",
      transaction_data["detail_id"].duplicated().sum())

print("\nUnique transaction detail IDs:",
      transaction_data["detail_id"].nunique())

print("\nMissing values in key analytical columns:")
key_columns = [
    "detail_id",
    "order_id",
    "customer_id",
    "product_id",
    "order_date",
    "full_date",
    "total_revenue",
    "quantity",
    "category"
]

print(transaction_data[key_columns].isna().sum())

Dataset dimensions: (25099, 49)

Duplicate transaction detail IDs: 0

Unique transaction detail IDs: 25099

Missing values in key analytical columns:
detail_id        0
order_id         0
customer_id      0
product_id       0
order_date       0
full_date        0
total_revenue    0
quantity         0
category         0
dtype: int64


# Final Integration Assessment Complete

|Check	|Result	|Status|
|---|---:|---|
|Dataset dimensions	|25,099 × 49	|Passed|
|Unique transaction detail IDs	|25,099	|Passed|
|Duplicate transaction detail IDs	|0	|Passed|
|Missing analytical fields	|0	|Passed|


We have successfully integrated the customer, order, product and date information without losing or duplicating transaction records.

In [28]:
# Save transaction_data table

transaction_data.to_csv("../01_Data/processed/transaction_data.csv")

# Building the Sales Table for Revenue and Profit

Our analysis needs one trustworthy table of sales with revenue, cost and profit. Before building it, we want to confirm the financial fields that profit depends on, and then apply the business rules agreed in the data quality log:

| Order situation | Counts as a sale? |
|---|---|
| Completed | Yes |
| Returned, return **rejected** | Yes, because the refund was refused and the customer kept the goods |
| Returned, return **refunded** or **pending**, or no return record | No |
| Cancelled | No |

## Financial integrity checks

We start from the cleaned tables and check that:

1. Order-line prices and costs match the product table.
2. There are no zero or negative prices, costs or quantities, and discounts fall within a sensible range.
3. Order lines still add up to each order's recorded revenue and cost.

In [29]:
import numpy as np

# Product prices and costs, as listed in the product table
line_check = order_details_clean.merge(
    products_clean[["product_id", "category", "unit_price", "unit_cost"]],
    on="product_id",
    how="left",
    suffixes=("", "_product"),
    validate="many_to_one"
)

print("Unit-price mismatches with product table:",
      (line_check["unit_price"] != line_check["unit_price_product"]).sum())
print("Unit-cost mismatches with product table:",
      (line_check["unit_cost"] != line_check["unit_cost_product"]).sum())

print("\nZero or negative unit prices:", (order_details_clean["unit_price"] <= 0).sum())
print("Zero or negative unit costs:", (order_details_clean["unit_cost"] <= 0).sum())
print("Quantity range:", order_details_clean["quantity"].min(),
      "to", order_details_clean["quantity"].max())
print("Discount values used (%):", sorted(order_details_clean["discount_pct"].unique()))

# Order lines vs order totals
line_totals = order_details_clean.groupby("order_id")[
    ["line_total_revenue", "line_total_cost"]
].sum()
reconciled = orders_clean.set_index("order_id").join(line_totals)

print("\nOrders where line revenue != order revenue:",
      ((reconciled["total_revenue"] - reconciled["line_total_revenue"]).abs() > 0.01).sum())
print("Orders where line cost != order cost:",
      ((reconciled["total_cost"] - reconciled["line_total_cost"]).abs() > 0.01).sum())

Unit-price mismatches with product table: 0
Unit-cost mismatches with product table: 0

Zero or negative unit prices: 0
Zero or negative unit costs: 0
Quantity range: 1 to 4
Discount values used (%): [np.int64(0), np.int64(5), np.int64(10), np.int64(15), np.int64(20), np.int64(25), np.int64(30)]

Orders where line revenue != order revenue: 0
Orders where line cost != order cost: 0


All checks pass. The financial fields are consistent across the order lines, the orders and the product table, so we can calculate profit directly from them.

### Margin available in each category

Before any discount, how much margin does each product carry?

In [30]:
products_margin = products_clean.assign(
    margin_before_discount=lambda d: (d["unit_price"] - d["unit_cost"]) / d["unit_price"]
)

products_margin.groupby("category")["margin_before_discount"].agg(
    products="size", min_margin="min", average_margin="mean", max_margin="max"
).round(3)

,products,min_margin,average_margin,max_margin
category,,,,
Clothing,308,0.500,0.522,0.600
Electronics,37,0.125,0.204,0.333


This is a key finding for the project:

- **Clothing** carries a 50–60% margin before discount.
- **Electronics** carries only 12.5–33%.

Discounts are applied to the selling price, so a 20–30% discount can remove most or all of an Electronics product's margin. We investigate this in the exploratory analysis and the loss-making order model.

## Applying the business rules

Each returned order gets the outcome of its return record. Rejected returns count as sales; refunded and pending returns, and the seven returned orders without a return record, do not.

In [31]:
orders_with_returns = orders_clean.merge(
    returns_clean[["order_id", "return_status"]],
    on="order_id",
    how="left",
    validate="one_to_one"
)

orders_with_returns["is_sale"] = (
    (orders_with_returns["order_status"] == "Completed")
    | ((orders_with_returns["order_status"] == "Returned")
       & (orders_with_returns["return_status"] == "Rejected"))
)

orders_with_returns["sale_treatment"] = np.select(
    [
        orders_with_returns["order_status"] == "Completed",
        orders_with_returns["order_status"] == "Cancelled",
        orders_with_returns["return_status"] == "Rejected",
        orders_with_returns["return_status"] == "Refunded",
        orders_with_returns["return_status"] == "Pending",
    ],
    [
        "Completed: sale",
        "Cancelled: excluded",
        "Return rejected: sale",
        "Refunded: excluded",
        "Return pending: excluded",
    ],
    default="Returned, no record: excluded",
)

(
    orders_with_returns
    .groupby("sale_treatment")
    .agg(orders=("order_id", "size"), order_value_egp=("total_revenue", "sum"))
    .sort_values("orders", ascending=False)
)

,orders,order_value_egp
sale_treatment,,
Completed: sale,10442,39652714.0
Refunded: excluded,819,3206996.5
Cancelled: excluded,495,1677927.5
Return pending: excluded,150,402529.5
Return rejected: sale,87,403586.5
"Returned, no record: excluded",7,7225.0


### Orders placed before customer registration

The data quality log found orders dated before the customer's registration date. We keep them as valid sales, because registration most likely records sign-up to the loyalty programme, which can happen after a first purchase. We flag them so the decision stays visible.

In [32]:
orders_with_returns = orders_with_returns.merge(
    customers_clean[["customer_id", "registration_date"]],
    on="customer_id",
    how="left",
    validate="many_to_one"
)
orders_with_returns["before_registration"] = (
    pd.to_datetime(orders_with_returns["order_date"])
    < pd.to_datetime(orders_with_returns["registration_date"])
)

pre_registration = orders_with_returns["before_registration"]
print("Orders before registration:", pre_registration.sum())
print(f"Share of order value: "
      f"{orders_with_returns.loc[pre_registration, 'total_revenue'].sum() / orders_with_returns['total_revenue'].sum():.1%}")

Orders before registration: 2034
Share of order value: 16.9%


Retaining these orders protects 17% of order value. Because registration dates are unreliable as a "first contact" date, customer tenure will be measured from each customer's **first purchase date** in later notebooks.

## Building the sales table

We keep one row per product line of every qualifying sale and add profit and margin. Product attributes come along so the table can be analysed on its own.

In [33]:
sale_orders = orders_with_returns.loc[
    orders_with_returns["is_sale"],
    ["order_id", "order_date", "customer_id", "store_id", "employee_id",
     "payment_method", "order_status", "return_status", "before_registration"]
]

sales_clean = (
    order_details_clean
    .merge(sale_orders, on="order_id", how="inner", validate="many_to_one")
    .merge(products_clean[["product_id", "category", "subcategory", "brand"]],
           on="product_id", how="left", validate="many_to_one")
    .rename(columns={"line_total_revenue": "line_revenue",
                     "line_total_cost": "line_cost"})
)

sales_clean["order_date"] = pd.to_datetime(sales_clean["order_date"])
sales_clean["line_profit"] = sales_clean["line_revenue"] - sales_clean["line_cost"]
sales_clean["line_margin"] = sales_clean["line_profit"] / sales_clean["line_revenue"]

column_order = [
    "detail_id", "order_id", "order_date", "customer_id", "store_id", "employee_id",
    "payment_method", "order_status", "return_status", "before_registration",
    "product_id", "category", "subcategory", "brand",
    "quantity", "unit_price", "discount_pct", "selling_price", "unit_cost",
    "line_revenue", "line_cost", "line_profit", "line_margin",
]
sales_clean = sales_clean[column_order].sort_values(["order_date", "order_id", "detail_id"])

print("Sales lines:", len(sales_clean))
print("Sales orders:", sales_clean["order_id"].nunique())
print("Customers with at least one sale:", sales_clean["customer_id"].nunique())
sales_clean.head()

Sales lines: 22075
Sales orders: 10529
Customers with at least one sale: 2908


,detail_id,order_id,order_date,customer_id,store_id,employee_id,payment_method,order_status,return_status,before_registration,...,brand,quantity,unit_price,discount_pct,selling_price,unit_cost,line_revenue,line_cost,line_profit,line_margin
3192,DTL0003680,ORD0001773,2022-01-01,CUS01029,2,EMP0047,Debit Card,Completed,NaN,True,...,Delta Wear,2,740,10,666.0,370,1332.0,740,592.0,0.444444
3473,DTL0004016,ORD0001947,2022-01-01,CUS02308,6,EMP0115,InstaPay,Completed,NaN,False,...,Lotus Classic,1,230,15,195.5,110,195.5,110,85.5,0.437340
3474,DTL0004017,ORD0001947,2022-01-01,CUS02308,6,EMP0115,InstaPay,Completed,NaN,False,...,Apple,1,15000,15,12750.0,12000,12750.0,12000,750.0,0.058824
5932,DTL0006806,ORD0003270,2022-01-01,CUS02303,7,EMP0128,Fawry,Completed,NaN,False,...,Delta Wear,1,360,10,324.0,170,324.0,170,154.0,0.475309
6962,DTL0007966,ORD0003830,2022-01-01,CUS01700,4,EMP0089,Debit Card,Completed,NaN,True,...,Delta Wear,2,120,5,114.0,60,228.0,120,108.0,0.473684


### Verifying the sales table

In [34]:
print("Duplicate detail IDs:", sales_clean["detail_id"].duplicated().sum())
print("Missing values in key columns:",
      sales_clean[["order_id", "customer_id", "product_id", "category",
                   "line_revenue", "line_cost"]].isna().sum().sum())

expected_lines = order_details_clean["order_id"].isin(
    orders_with_returns.loc[orders_with_returns["is_sale"], "order_id"]
).sum()
print("Lines expected from qualifying orders:", expected_lines,
      "| lines in sales table:", len(sales_clean))

customers_without_sales = (~customers_clean["customer_id"].isin(sales_clean["customer_id"])).sum()
print("Registered customers with no qualifying sale:", customers_without_sales)

Duplicate detail IDs: 0
Missing values in key columns: 0
Lines expected from qualifying orders: 22075 | lines in sales table: 22075
Registered customers with no qualifying sale: 92


Every qualifying order line is present exactly once.

Registered customers with no qualifying sale stay in `customers_clean`. They will be reported separately as **"Registered, never purchased"** and kept out of the clustering. This group is slightly larger than the 55 customers with no orders at all, because a few customers' only orders were cancelled or refunded.

### Revenue and profit summary

In [35]:
summary = sales_clean.groupby("category").agg(
    revenue=("line_revenue", "sum"),
    cost=("line_cost", "sum"),
    profit=("line_profit", "sum"),
)
summary.loc["Total"] = summary.sum()
summary["margin"] = summary["profit"] / summary["revenue"]
summary["revenue_share"] = summary["revenue"] / summary.loc["Total", "revenue"]
summary["profit_share"] = summary["profit"] / summary.loc["Total", "profit"]

summary.style.format({
    "revenue": "EGP {:,.0f}", "cost": "EGP {:,.0f}", "profit": "EGP {:,.0f}",
    "margin": "{:.1%}", "revenue_share": "{:.1%}", "profit_share": "{:.1%}",
})

,revenue,cost,profit,margin,revenue_share,profit_share
category,,,,,,
Clothing,"EGP 9,603,760","EGP 5,046,060","EGP 4,557,700",47.5%,24.0%,53.5%
Electronics,"EGP 30,452,540","EGP 26,491,300","EGP 3,961,240",13.0%,76.0%,46.5%
Total,"EGP 40,056,300","EGP 31,537,360","EGP 8,518,940",21.3%,100.0%,100.0%


In [36]:
order_profit = sales_clean.groupby("order_id")["line_profit"].sum()
pending_value = orders_with_returns.loc[
    orders_with_returns["return_status"] == "Pending", "total_revenue"
].sum()

print("Loss-making sales orders:", (order_profit < 0).sum(),
      f"({(order_profit < 0).mean():.1%} of sales orders)")
print(f"Profit lost on loss-making orders: EGP {order_profit[order_profit < 0].sum():,.0f}")
print(f"\nSensitivity: pending returns are worth EGP {pending_value:,.0f}, "
      f"{pending_value / summary.loc['Total', 'revenue']:.1%} of qualifying revenue")

Loss-making sales orders: 126 (1.2% of sales orders)
Profit lost on loss-making orders: EGP -111,176

Sensitivity: pending returns are worth EGP 402,530, 1.0% of qualifying revenue


**What the summary tells us**

- Electronics bring in most of the revenue but a much smaller share of the profit, because of their thinner margins.
- Only about 1% of sales orders lose money. These rare orders are the target of the loss-making order model.
- Even if every pending return were later rejected, revenue would change by about 1%, so the cautious treatment doesn't affect any finding.

### Saving the sales table

We save without the DataFrame index so the file has no unnamed index column.

In [37]:
sales_clean.to_csv("../01_Data/processed/sales_clean.csv", index=False)
print("Saved sales_clean.csv:", sales_clean.shape)

Saved sales_clean.csv: (22075, 23)


# Cleaning Complete

|Output	|Grain	|Purpose|
|---|---|---|
|`customers_clean.csv`	|One row per customer	|Customer attributes for segmentation|
|`orders_clean.csv`	|One row per order	|All orders, every status|
|`returns_clean.csv`	|One row per return	|Return outcomes|
|`transaction_data.csv`	|One row per order line, every status	|Integrated reference table|
|`sales_clean.csv`	|One row per order line of a **qualifying sale**	|**Main table for revenue, profit and margin analysis**|

From here on, every revenue, profit and margin figure in the project comes from `sales_clean.csv`.